# 01 - Why Spark and Working in Fabric

This opening lesson introduces the small set of Spark ideas that will make the remaining notebooks easier to understand.

## Learning objectives

By the end of this notebook, you will be able to:

- run cells in a Fabric notebook;
- describe the roles of the driver, executors, and partitions;
- distinguish a transformation from an action; and
- recognise why shuffles and `collect()` require care.

### Intro
No previous PySpark knowledge is required. Use a **Spark / PySpark** notebook.

Run notebook cells from top to bottom with the play button or `Shift+Enter`. Variables created in one cell remain available to later cells while the session is active.

In [ ]:
print(f'Spark version: {spark.version}')
spark.range(5).show()

## Why Spark?

Spark processes data across multiple workers when a single-machine tool is no longer the right fit. It is useful for repeatable ingestion, transformation, SQL analytics, streaming, and distributed feature preparation.

Use pandas, Polars, or a SQL engine when the data fits comfortably on one machine.

## The execution model

`notebook code -> driver -> tasks on executors -> data partitions`

- The **driver** runs the notebook code and coordinates the work.
- **Executors** are worker processes that perform tasks.
- A **partition** is a slice of the data that one task can process.

More partitions are not automatically better: tiny partitions add scheduling overhead, while very large partitions can limit parallelism.

## Transformations, actions, and lazy evaluation

A transformation such as `select`, `filter`, or `withColumn` describes a new DataFrame. Spark waits before scanning the data so it can optimise the combined plan.

An action such as `show()`, `count()`, `collect()`, or a write asks Spark to execute that plan.

In [ ]:
from pyspark.sql import functions as F

numbers = spark.range(10)
even_numbers = numbers.filter((F.col('id') % 2) == 0)  # Transformation

even_numbers.explain('formatted')  # Inspect the plan
even_numbers.show()                # Execute and return a small preview

## Data movement, reuse, and safety

- `groupBy`, many joins, `distinct`, and sorting often cause a **shuffle**, which moves records between executors.
- Use `cache()` or `persist()` only when an expensive DataFrame will be reused, and call `unpersist()` afterwards.
- Spark records transformation lineage so lost partitions can usually be recomputed.
- `collect()` brings every result row to the driver. Use it only when the result is known to be small.

## Quick check

1. Which line in the example is the first action?
2. Why would collecting five rows be safe while collecting millions might not be?
3. Which later course operations are likely to shuffle data?

### Expected result

You should identify `show()` as the first action that returns data, explain that `collect()` uses driver memory, and name grouping, joins, deduplication, or sorting as likely shuffle operations.

### Solution - reveal after attempting

1. `even_numbers.show()` executes the plan and returns the preview. `explain()` inspects the plan rather than returning the dataset.
2. Five rows fit safely in driver memory; millions may exhaust it.
3. `groupBy`, many joins, `distinct` / deduplication, and global sorting commonly require a shuffle.

## Key takeaway

Spark builds a plan on the driver and executes tasks over partitions. Keep large results distributed and be alert to operations that move data.

**Next:** create and inspect Spark DataFrames.